In [ ]:
import pandas as pd
from pathlib import Path
import json
from datetime import datetime, timezone

# Paths

DATA_PATH = "../backend/data/D1_clean_all_hives_15min.parquet"
OUT_DIR   = Path("../backend/data/splits_70_15_15")
OUT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_PARQUET = OUT_DIR / "train.parquet"
VAL_PARQUET   = OUT_DIR / "val.parquet"
TEST_PARQUET  = OUT_DIR / "test.parquet"

SKIPPED_HIVES_CSV = OUT_DIR / "skipped_hives.csv"
SPLIT_SUMMARY_CSV = OUT_DIR / "split_summary.csv"
SPLIT_META_JSON   = OUT_DIR / "split_metadata.json"

EXPORT_CSV = False
TRAIN_CSV = OUT_DIR / "train.csv"
VAL_CSV   = OUT_DIR / "val.csv"
TEST_CSV  = OUT_DIR / "test.csv"

# Split settings

TRAIN_FRAC = 0.70
VAL_FRAC   = 0.15
TEST_FRAC  = 0.15
MIN_SAMPLES_PER_HIVE = 50   # minimum unique timestamps per hive

assert abs(TRAIN_FRAC + VAL_FRAC + TEST_FRAC - 1.0) < 1e-9, "Fractions must sum to 1.0"

# NOTE: This split is deterministic time-based; SEED not required.
SEED = 42


# Load

print("Loading:", DATA_PATH)
df = pd.read_parquet(DATA_PATH)

required = ["tag_number", "published_at"]
missing_required = [c for c in required if c not in df.columns]
assert not missing_required, f"Missing required columns: {missing_required}"

df["published_at"] = pd.to_datetime(df["published_at"], utc=True, errors="coerce")
df = df.dropna(subset=["published_at"]).copy()

# Sort for time-series correctness
df = df.sort_values(["tag_number", "published_at"]).reset_index(drop=True)

print("Rows:", len(df), "| Hives:", df["tag_number"].nunique())


# Split per hive (TIMESTAMP-BASED)

train_parts, val_parts, test_parts = [], [], []
skipped = []
split_rows = []

for hive_id, g in df.groupby("tag_number", sort=True):
    g = g.sort_values("published_at").reset_index(drop=True)

    # Use unique timestamps for splitting (robust)
    ts = g["published_at"].drop_duplicates().sort_values().reset_index(drop=True)
    n_ts = len(ts)

    if n_ts < MIN_SAMPLES_PER_HIVE:
        skipped.append({"tag_number": str(hive_id), "n_unique_timestamps": int(n_ts), "reason": "too_few_samples"})
        continue

    # Determine cut indices on unique timestamps
    train_end = int(n_ts * TRAIN_FRAC)
    val_end   = int(n_ts * (TRAIN_FRAC + VAL_FRAC))

    # Need at least 1 timestamp in each split
    if train_end < 1 or (val_end - train_end) < 1 or (n_ts - val_end) < 1:
        skipped.append({"tag_number": str(hive_id), "n_unique_timestamps": int(n_ts), "reason": "split_too_small"})
        continue

    # Cut timestamps (use inclusive/exclusive boundaries to avoid overlap)
    train_cut = ts.iloc[train_end - 1]     # last ts in train
    val_cut   = ts.iloc[val_end - 1]       # last ts in val

    train_g = g[g["published_at"] <= train_cut].copy()
    val_g   = g[(g["published_at"] > train_cut) & (g["published_at"] <= val_cut)].copy()
    test_g  = g[g["published_at"] > val_cut].copy()

    # Final guard 
    if len(train_g) == 0 or len(val_g) == 0 or len(test_g) == 0:
        skipped.append({"tag_number": str(hive_id), "n_unique_timestamps": int(n_ts), "reason": "empty_split_after_cut"})
        continue

    split_rows.append({
        "tag_number": str(hive_id),
        "n_rows_total": int(len(g)),
        "n_unique_timestamps": int(n_ts),
        "n_train_rows": int(len(train_g)),
        "n_val_rows": int(len(val_g)),
        "n_test_rows": int(len(test_g)),
        "train_start": str(train_g["published_at"].min()),
        "train_end":   str(train_g["published_at"].max()),
        "val_start":   str(val_g["published_at"].min()),
        "val_end":     str(val_g["published_at"].max()),
        "test_start":  str(test_g["published_at"].min()),
        "test_end":    str(test_g["published_at"].max()),
    })

    train_parts.append(train_g)
    val_parts.append(val_g)
    test_parts.append(test_g)

if len(train_parts) == 0:
    raise ValueError("No hives were kept after filtering. Lower MIN_SAMPLES_PER_HIVE or check input data.")

train_df = pd.concat(train_parts, ignore_index=True)
val_df   = pd.concat(val_parts, ignore_index=True)
test_df  = pd.concat(test_parts, ignore_index=True)

# Final sorting
train_df = train_df.sort_values(["tag_number", "published_at"]).reset_index(drop=True)
val_df   = val_df.sort_values(["tag_number", "published_at"]).reset_index(drop=True)
test_df  = test_df.sort_values(["tag_number", "published_at"]).reset_index(drop=True)

print("\nSplit sizes:")
print("Train:", train_df.shape)
print("Val  :", val_df.shape)
print("Test :", test_df.shape)


# leakage checks

train_max = train_df.groupby("tag_number")["published_at"].max()
val_min   = val_df.groupby("tag_number")["published_at"].min()
val_max   = val_df.groupby("tag_number")["published_at"].max()
test_min  = test_df.groupby("tag_number")["published_at"].min()

common_hives = train_max.index.intersection(val_min.index).intersection(val_max.index).intersection(test_min.index)

# Ensure train before val, and val before test (no overlap)
check = (train_max.loc[common_hives] < val_min.loc[common_hives]) & (val_max.loc[common_hives] < test_min.loc[common_hives])

n_hives_checked = int(len(common_hives))
n_pass = int(check.sum())
n_fail = n_hives_checked - n_pass

print(f"\nLeakage check per hive: max(train) < min(val) AND max(val) < min(test)")
print(f"PASS: {n_pass}/{n_hives_checked} hives")

if n_fail > 0:
    bad_hives = check[~check].index.astype(str).tolist()
    print("First 10 failing hives:", bad_hives[:10])
    raise ValueError(f"Time leakage detected for {len(bad_hives)} hives.")

# Ensure hive sets match (all hives appear in all splits)
assert set(train_df["tag_number"]) == set(val_df["tag_number"]) == set(test_df["tag_number"]), \
    "Mismatch: some hives are missing in val/test. Check MIN_SAMPLES_PER_HIVE or split boundaries."


# Save outputs

train_df.to_parquet(TRAIN_PARQUET, index=False)
val_df.to_parquet(VAL_PARQUET, index=False)
test_df.to_parquet(TEST_PARQUET, index=False)

print("\nSaved:")
print(" -", TRAIN_PARQUET)
print(" -", VAL_PARQUET)
print(" -", TEST_PARQUET)

# Save skipped hives
skipped_df = pd.DataFrame(skipped)
skipped_df.to_csv(SKIPPED_HIVES_CSV, index=False)
print(" -", SKIPPED_HIVES_CSV, f"(skipped {len(skipped_df)} hives)")

# Save per-hive split summary
split_summary = pd.DataFrame(split_rows)
split_summary.to_csv(SPLIT_SUMMARY_CSV, index=False)
print(" -", SPLIT_SUMMARY_CSV)

# Save split metadata JSON
meta = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "seed": int(SEED),  # stored for completeness; split is deterministic
    "split_type": "per_hive_time_order_timestamp_based",
    "split_ratios": {"train": float(TRAIN_FRAC), "val": float(VAL_FRAC), "test": float(TEST_FRAC)},
    "min_samples_per_hive_unique_timestamps": int(MIN_SAMPLES_PER_HIVE),
    "n_hives_total_in_input": int(df["tag_number"].nunique()),
    "n_hives_kept": int(train_df["tag_number"].nunique()),
    "n_hives_skipped": int(len(skipped)),
    "rows_total_kept": int(len(train_df) + len(val_df) + len(test_df)),
    "rows_train": int(len(train_df)),
    "rows_val": int(len(val_df)),
    "rows_test": int(len(test_df)),
}

with open(SPLIT_META_JSON, "w", encoding="utf-8") as f:
    json.dump(meta, f, indent=2)

print(" -", SPLIT_META_JSON)

# Optional CSV exports
if EXPORT_CSV:
    train_df.to_csv(TRAIN_CSV, index=False)
    val_df.to_csv(VAL_CSV, index=False)
    test_df.to_csv(TEST_CSV, index=False)
    print("\nAlso exported CSV:")
    print(" -", TRAIN_CSV)
    print(" -", VAL_CSV)
    print(" -", TEST_CSV)

# Quick sanity stats

kept_hives = train_df["tag_number"].nunique()
print("\nKept hives:", kept_hives)

total_kept = (len(train_df) + len(val_df) + len(test_df))
print("Train fraction (actual):", len(train_df) / total_kept)
print("Val fraction (actual):  ", len(val_df) / total_kept)
print("Test fraction (actual): ", len(test_df) / total_kept)

Loading: ../backend/data/D1_clean_all_hives_15min.parquet
Rows: 801080 | Hives: 52

Split sizes:
Train: (560733, 31)
Val  : (120160, 31)
Test : (120187, 31)

Leakage check per hive: max(train) < min(val) AND max(val) < min(test)
PASS: 52/52 hives

Saved:
 - ..\backend\data\splits_70_15_15\train.parquet
 - ..\backend\data\splits_70_15_15\val.parquet
 - ..\backend\data\splits_70_15_15\test.parquet
 - ..\backend\data\splits_70_15_15\skipped_hives.csv (skipped 0 hives)
 - ..\backend\data\splits_70_15_15\split_summary.csv
 - ..\backend\data\splits_70_15_15\split_metadata.json

Kept hives: 52
Train fraction (actual): 0.6999712887601738
Val fraction (actual):   0.1499975033704499
Test fraction (actual):  0.15003120786937635
